# Joint-document token analysis & truncation explorer

Inspect the token-length distribution of the SMARTEHR joint documents (whole-history text
per patient) and preview exactly what `extract_qwen_embeddings_longitudinal.py --joint`
produces at different truncation thresholds — so you can size `--max-tokens-per-block`
and `--max-seq-length` from data instead of guessing (and avoid the OOMs).

**Prereqs**
- Run from the repo root (`jupyter` launched there), so `scripts/` and the CWD-relative paths resolve.
- `transformers` new enough to load the encoder tokenizer; point `TEXT_DIR` at the **enriched** Stage-2 build.


In [ ]:
import sys
sys.path.append(".")  # run from repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer

# ---------------- CONFIG ----------------
TEXT_DIR = "data/dummy_data/longitudinal_dummy_smart_survival_longitudinal"  # ENRICHED Stage-2 text build
SPLIT = "train"
TOKENIZER_NAME = "Qwen/Qwen3-Embedding-0.6B"   # the encoder you extract with
SAMPLE_N = None   # set to e.g. 1000 to subsample patients for a faster pass


In [ ]:
tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

def n_tokens(text: str) -> int:
    return len(tok(text, add_special_tokens=False)["input_ids"])

def truncate_to_tokens(text: str, n: int) -> str:
    ids = tok(text, add_special_tokens=False, truncation=True, max_length=n)["input_ids"]
    return tok.decode(ids, skip_special_tokens=True)

# Reuse the SAME joint-document builder the extractor uses, so previews match extraction exactly.
try:
    from scripts.smartehr.extract_qwen_embeddings_longitudinal import build_joint_document, _time_phrase
    print("Using build_joint_document from the extractor (previews match extraction).")
except Exception as e:
    print("Import failed, using a local copy of build_joint_document:", e)
    def _time_phrase(dt):
        y = abs(float(dt))
        if y < 1.0:
            m = max(1, round(y * 12)); return f"about {m} month{'s' if m != 1 else ''} before baseline"
        return f"{y:.1f} years before baseline"
    def build_joint_document(texts, time_deltas, exclude_baseline=False, max_tokens_per_block=None, truncate_fn=None):
        blocks = []
        for i, (txt, dt) in enumerate(zip(texts, time_deltas)):
            if i == 0:
                if exclude_baseline: continue
                title = "At baseline (enrollment)"
            else:
                title = _time_phrase(dt)
            if max_tokens_per_block and truncate_fn is not None:
                txt = truncate_fn(txt, max_tokens_per_block)
            blocks.append(f"[{title}]\n{txt}")
        return "\n\n".join(blocks) if blocks else "No clinical records before baseline."


In [ ]:
ds = load_dataset("parquet", data_files=f"{TEXT_DIR}/{SPLIT}.parquet", split="train")
if SAMPLE_N:
    ds = ds.select(range(min(SAMPLE_N, len(ds))))
print(f"{len(ds):,} patients from {TEXT_DIR}/{SPLIT}.parquet")


## 1. Token counts per block (time point) and per patient

In [ ]:
block_rows, patient_rows = [], []
for pidx, rec in enumerate(ds):
    texts, deltas = rec["texts"], rec["time_deltas_list"]
    blens = [n_tokens(t) for t in texts]
    for bidx, (d, bl) in enumerate(zip(deltas, blens)):
        block_rows.append(dict(patient=pidx, block=bidx, is_baseline=(bidx == 0),
                               time_delta_years=float(d), n_tokens=bl))
    patient_rows.append(dict(patient=pidx,
                             n_timepoints=len(texts),
                             n_events=len(texts) - 1,
                             baseline_tokens=blens[0],
                             event_tokens=int(sum(blens[1:])),
                             total_block_tokens=int(sum(blens)),   # ~ untruncated joint doc (excl. titles)
                             max_block_tokens=int(max(blens))))
blocks_df = pd.DataFrame(block_rows)
pat_df = pd.DataFrame(patient_rows)

print("Per-block token stats:")
display(blocks_df.n_tokens.describe(percentiles=[.5, .9, .95, .99]).to_frame("tokens/block"))
print("Per-patient stats:")
display(pat_df[["n_timepoints", "total_block_tokens", "max_block_tokens"]]
        .describe(percentiles=[.5, .9, .95, .99]))


In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 9))

ax[0, 0].hist(blocks_df.n_tokens, bins=60)
ax[0, 0].set(title="Tokens per block (time point)", xlabel="tokens", ylabel="# blocks")

ax[0, 1].hist(pat_df.total_block_tokens, bins=60)
ax[0, 1].set(title="Tokens per patient (untruncated joint doc, approx)", xlabel="tokens", ylabel="# patients")

mx = int(pat_df.n_timepoints.max())
ax[1, 0].hist(pat_df.n_timepoints, bins=range(1, mx + 2))
ax[1, 0].set(title="Time points per patient", xlabel="# time points", ylabel="# patients")

ax[1, 1].hist([blocks_df[blocks_df.is_baseline].n_tokens,
               blocks_df[~blocks_df.is_baseline].n_tokens],
              bins=40, label=["baseline", "event"])
ax[1, 1].legend(); ax[1, 1].set(title="Tokens per block: baseline vs event", xlabel="tokens")

plt.tight_layout(); plt.show()


In [ ]:
# Longest blocks — usually the clinical-note time points (consult/radiology/discharge letters).
top = blocks_df.sort_values("n_tokens", ascending=False).head(10)
for _, r in top.iterrows():
    txt = ds[int(r.patient)]["texts"][int(r.block)]
    print(f"patient {int(r.patient)} block {int(r.block)} ({r.n_tokens} tok): {txt[:160]!r}")


## 2. Truncation-threshold explorer\nHow `--max-tokens-per-block` shapes the joint-doc length and how much content it keeps. The per-patient doc length rebuilt below matches extraction exactly (same builder + tokenizer).

In [ ]:
CAPS = [16, 32, 48, 64, 96, 128]

# (a) content-level impact — computed from block lengths (fast, no re-tokenization)
b = blocks_df.n_tokens.values
impact = pd.DataFrame([
    dict(cap=c,
         pct_blocks_truncated=round(100 * (b > c).mean(), 1),
         pct_content_kept=round(100 * np.minimum(b, c).sum() / b.sum(), 1))
    for c in CAPS
])
print("Per-block cap: how many blocks get trimmed, and what fraction of total content survives")
display(impact)


In [ ]:
# (b) resulting joint-doc length per patient at each cap (exact: rebuild + tokenize)
def doc_lengths_at_cap(cap):
    return np.array([
        n_tokens(build_joint_document(r["texts"], r["time_deltas_list"],
                                      max_tokens_per_block=cap, truncate_fn=truncate_to_tokens))
        for r in ds
    ])

lengths = {c: doc_lengths_at_cap(c) for c in CAPS}
lengths[None] = pat_df.total_block_tokens.values  # untruncated (approx, excl. titles)

summary = pd.DataFrame({
    (f"cap={c}" if c is not None else "untrunc"): pd.Series(v).describe(percentiles=[.5, .9, .95, .99, 1.0])
    for c, v in lengths.items()
}).loc[["mean", "50%", "90%", "95%", "99%", "max"]]
print("Joint-doc tokens per patient (pick a cap whose 'max' fits your memory budget):")
display(summary.round(0))

plt.figure(figsize=(11, 5))
for c in CAPS:
    plt.hist(lengths[c], bins=60, histtype="step", label=f"cap={c}")
plt.legend(); plt.xlabel("joint doc tokens / patient"); plt.ylabel("# patients")
plt.title("Joint-doc length vs --max-tokens-per-block"); plt.show()


## 3. Preview a patient's joint document at chosen thresholds\nExactly what the encoder will see. Tune `MAX_TOKENS_PER_BLOCK` / `MAX_SEQ_LENGTH` and re-run.

In [ ]:
def show_patient(idx, max_tokens_per_block=None, max_seq_length=None, exclude_baseline=False):
    rec = ds[idx]
    full = build_joint_document(rec["texts"], rec["time_deltas_list"], exclude_baseline=exclude_baseline)
    doc = build_joint_document(rec["texts"], rec["time_deltas_list"], exclude_baseline=exclude_baseline,
                               max_tokens_per_block=max_tokens_per_block, truncate_fn=truncate_to_tokens)
    if max_seq_length:
        doc = truncate_to_tokens(doc, max_seq_length)
    print(f"patient {idx}: {len(rec['texts'])} time points")
    print(f"  full doc      : {n_tokens(full):>6} tokens, {full.count(chr(91))} blocks")
    print(f"  after trunc   : {n_tokens(doc):>6} tokens, {doc.count(chr(91))} blocks   "
          f"(per-block={max_tokens_per_block}, max-seq={max_seq_length})")
    print("=" * 90)
    print(doc)

# find a patient with a long history to make the effect visible
idx = int(pat_df.sort_values("total_block_tokens", ascending=False).iloc[0].patient)
show_patient(idx, max_tokens_per_block=48, max_seq_length=4096)


## Choosing thresholds

- **`--max-tokens-per-block`**: read the `max` column in section 2(b) for each cap — pick the largest cap
  whose `max` (worst-case patient) comfortably fits memory. Per-block capping keeps **every** time point
  (breadth), trimming only long clinical-note blocks. Section 2(a) shows how much content each cap keeps.
- **`--max-seq-length`**: a hard backstop for the rare very-long-history patient. Set it above the `95–99%`
  doc length so it only clips outliers (which lose their **oldest** blocks — acceptable).
- If section 1 shows a few blocks with thousands of tokens (clinical notes), a small per-block cap (32–64)
  recovers most breadth cheaply. Re-run section 3 on the longest-history patient to sanity-check the output.
